# Lezione 11 - Protocollo di Contesto del Modello (MCP)

Il **Protocollo di Contesto del Modello (MCP)** è uno standard aperto che consente agli agenti di scoprire e utilizzare dinamicamente strumenti, risorse e fonti di dati durante l'esecuzione. Invece di incorporare gli strumenti direttamente in un agente, MCP permette agli agenti di connettersi a server esterni che espongono capacità su richiesta.

In questa lezione imparerai:
- Che cos'è MCP e perché è importante per i sistemi basati su agenti
- Come funziona l'architettura client-server di MCP
- Come costruire agenti che utilizzano la scoperta degli strumenti in stile MCP


## Configurazione

**Prerequisiti:**
- Progetto Azure AI Foundry con un modello distribuito
- Esegui `az login` per l'autenticazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Che cos'è il Model Context Protocol (MCP)?

MCP definisce un modo standard per gli agenti AI di scoprire e interagire con strumenti esterni e fonti di dati:

- **MCP Server**: Espone strumenti, risorse e prompt tramite un protocollo standard
- **MCP Client**: Il runtime dell'agente che si connette ai server e scopre le capacità disponibili
- **Dynamic Discovery**: Gli agenti non hanno bisogno di strumenti codificati rigidamente — scoprono cosa è disponibile a runtime

Questo è potente per costruire sistemi di agenti estendibili in cui nuove capacità possono essere aggiunte senza modificare il codice dell'agente.


## Come funziona MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. L'agente (MCP client) si connette a un MCP server
2. Il server risponde con un elenco di strumenti disponibili e dei loro schemi
3. L'agente può quindi chiamare qualsiasi strumento scoperto durante il suo ragionamento
4. I risultati ritornano attraverso lo stesso protocollo


## Simulazione della scoperta degli strumenti MCP

Poiché un vero server MCP richiede un processo server in esecuzione, dimostreremo il modello usando le funzioni `@tool` che simulano ciò che un servizio di alloggio connesso a MCP fornirebbe.

In produzione, questi strumenti verrebbero scoperti dinamicamente da un server MCP anziché definiti localmente.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Costruire un agente con strumenti in stile MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP in Produzione

In un ambiente di produzione, MCP abilita modelli potenti:

- **Scoperta dinamica degli strumenti**: Gli agenti si connettono ai server MCP e scoprono gli strumenti a runtime
- **Architettura disaccoppiata**: I fornitori di strumenti possono aggiornare i propri strumenti indipendentemente dall'agente
- **Condivisione tra organizzazioni**: I team possono esporre capacità tramite server MCP che qualsiasi agente può utilizzare
- **Supporto del Microsoft Agent Framework**: MAF ha un supporto client MCP integrato tramite l'integrazione `mcp`

Per utilizzare un server MCP reale con MAF, ci si connetterebbe tramite `hosted_mcp_tool()` o l'integrazione client MCP.

**Scopri di più:**
- [Specifiche MCP](https://modelcontextprotocol.io/)
- [Supporto MCP del Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Riepilogo

In questa lezione, hai imparato:
- **MCP** è uno standard aperto per la scoperta dinamica degli strumenti tra agenti e fornitori di strumenti
- La **architettura client-server** consente agli agenti di scoprire le capacità in fase di esecuzione
- MCP consente **sistemi di agenti estendibili e disaccoppiati** in cui gli strumenti possono essere aggiunti senza modifiche al codice
- Microsoft Agent Framework fornisce **supporto MCP integrato** per l'uso in produzione


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Dichiarazione di non responsabilità:
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica Co-op Translator (https://github.com/Azure/co-op-translator). Sebbene ci impegniamo a garantire l'accuratezza, si prega di notare che le traduzioni automatiche possono contenere errori o imprecisioni. Il documento originale nella lingua di origine deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un traduttore umano. Non siamo responsabili per eventuali incomprensioni o interpretazioni errate derivanti dall'uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
